In [ ]:
from huggingface_hub import login
login()

In [ ]:
from dataclasses import dataclass


@dataclass
class ExperimentConfig:
    model_id: str = "meta-llama/Llama-3.2-1B-Instruct"
    max_new_tokens: int = 80
    top_p: float = 0.95
    runs_per_condition: int = 5
    output_dir: str = "results"


PROMPT_VARIANTS = [
    {
        "prompt_id": "clean",
        "perturbation_type": "none",
        "prompt": "Explain unlearning in one sentence.",
    },
    {
        "prompt_id": "typo",
        "perturbation_type": "typo",
        "prompt": "Explain unlearning in one sentnce.",
    },
    {
        "prompt_id": "numeric",
        "perturbation_type": "wording",
        "prompt": "Explain unlearning in 1 sentence.",
    },
    {
        "prompt_id": "uppercase",
        "perturbation_type": "formatting",
        "prompt": "EXPLAIN unlearning in one sentence.",

    },
    {
        "prompt_id": "punctuation",
        "perturbation_type": "punctuation",
        "prompt": "Explain unlearning in one sentence!!",
    },
    {
        "prompt_id": "polite",
        "perturbation_type": "tone",
        "prompt": "Please explain unlearning in one sentence.",
    },
    {
        "prompt_id": "compressed",
        "perturbation_type": "compressed",
        "prompt": "Unlearning, one sentence.",
    }
]

TEMPERATURES = [0.2, 0.7, 1.2]

In [ ]:
import os

def ensure_output_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_id: str):
    print(f"Loading tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print(f"Loading model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto"
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    return tokenizer, model

In [ ]:
def classify_behavior(text: str) -> str:
    stripped = text.strip()
    lower = stripped.lower()
    words = stripped.split()

    if not stripped:
        return "empty"

    if "?" in stripped or lower.startswith("can you") or "would you like" in lower:
        return "conversational_extension"

    if any(marker in lower for marker in ["for example", "instance", "e.g."]):
        return "example_extension"

    if len(words) <= 25:
        return "compact_definition"

    if len(words) >= 65:
        return "verbose_definition"

    return "standard_definition"

In [ ]:
def lexical_diversity(text: str) -> float:
    words = [w.strip(".,!?;:()[]{}\"'").lower() for w in text.split()]
    words = [w for w in words if w]

    if not words:
        return 0.0

    return len(set(words)) / len(words)

In [1]:
def sentence_count(text: str) -> int:
    count = sum(text.count(p) for p in [".", "!", "?"])
    return max(count, 1 if text.strip() else 0)

In [ ]:
def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
import time
from typing import Dict, Any

import math
import torch

def generate_once(
    tokenizer,
    model,
    prompt: str,
    temperature: float,
    top_p: float,
    max_new_tokens: int
) -> Dict[str, Any]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_tokens = inputs["input_ids"].shape[-1]

    sync_cuda()
    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    sync_cuda()
    elapsed = time.perf_counter() - start

    total_tokens = output_ids.shape[-1]
    generated_tokens = total_tokens - input_tokens
    tokens_per_second = generated_tokens / elapsed if elapsed > 0 else math.nan

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if full_text.startswith(prompt):
        generated_text = full_text[len(prompt):].strip()
    else:
        generated_text = full_text.strip()

    return {
        "input_tokens": input_tokens,
        "total_tokens": total_tokens,
        "generated_tokens": generated_tokens,
        "elapsed_seconds": elapsed,
        "tokens_per_second": tokens_per_second,
        "full_text": full_text,
        "generated_text": generated_text
    }

In [ ]:
import pandas as pd

from typing import List

def run_experiment(config: ExperimentConfig) -> pd.DataFrame:
    ensure_output_dir(config.output_dir)

    tokenizer, model = load_model(config.model_id)

    print("\nDevice info")
    print("-" * 60)
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {round(props.total_memory / (1024 ** 3), 2)} GB")
    else:
        print(f"Model device: {model.device}")

    print("\nStarting trajectory divergence experiment")
    print("-" * 60)

    records: List[Dict[str, Any]] = []

    # Warmup
    warmup_prompt = PROMPT_VARIANTS[0]["prompt"]
    warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model.generate(
            **warmup_inputs,
            max_new_tokens=5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    total_conditions = len(PROMPT_VARIANTS) * len(TEMPERATURES) * config.runs_per_condition
    completed = 0

    for prompt_info in PROMPT_VARIANTS:
        for temperature in TEMPERATURES:
            for run_id in range(config.runs_per_condition):
                completed += 1
                print(
                    f"[{completed}/{total_conditions}] "
                    f"prompt={prompt_info['prompt_id']} "
                    f"temp={temperature} "
                    f"run={run_id}"
                )

                result = generate_once(
                    tokenizer=tokenizer,
                    model=model,
                    prompt=prompt_info["prompt"],
                    temperature=temperature,
                    top_p=config.top_p,
                    max_new_tokens=config.max_new_tokens
                )

                generated_text = result["generated_text"]

                record = {
                    "model_id": config.model_id,
                    "prompt_id": prompt_info["prompt_id"],
                    "perturbation_type": prompt_info["perturbation_type"],
                    "prompt": prompt_info["prompt"],
                    "temperature": temperature,
                    "top_p": config.top_p,
                    "run_id": run_id,
                    **result,
                    "word_count": len(generated_text.split()),
                    "char_count": len(generated_text),
                    "sentence_count": sentence_count(generated_text),
                    "lexical_diversity": lexical_diversity(generated_text),
                    "behavior_label": classify_behavior(generated_text)
                }

                records.append(record)

    df = pd.DataFrame(records)
    return df

In [ ]:
def create_summaries(df: pd.DataFrame, output_dir: str) -> None:
    generations_path = os.path.join(output_dir, "generations.csv")
    df.to_csv(generations_path, index=False)

    summary_by_temperature = (
        df.groupby("temperature")
        .agg(
            n_generations=("generated_text", "count"),
            avg_tokens_per_second=("tokens_per_second", "mean"),
            std_tokens_per_second=("tokens_per_second", "std"),
            avg_generated_tokens=("generated_tokens", "mean"),
            avg_word_count=("word_count", "mean"),
            avg_sentence_count=("sentence_count", "mean"),
            avg_lexical_diversity=("lexical_diversity", "mean"),
        )
        .reset_index()
    )

    summary_by_prompt = (
        df.groupby(["prompt_id", "perturbation_type", "temperature"])
        .agg(
            n_generations=("generated_text", "count"),
            avg_tokens_per_second=("tokens_per_second", "mean"),
            avg_word_count=("word_count", "mean"),
            avg_lexical_diversity=("lexical_diversity", "mean"),
        )
        .reset_index()
    )

    behavior_label_summary = (
        df.groupby(["temperature", "behavior_label"])
        .size()
        .reset_index(name="count")
        .sort_values(["temperature", "count"], ascending=[True, False])
    )

    summary_by_temperature.to_csv(
        os.path.join(output_dir, "summary_by_temperature.csv"),
        index=False
    )

    summary_by_prompt.to_csv(
        os.path.join(output_dir, "summary_by_prompt.csv"),
        index=False
    )

    behavior_label_summary.to_csv(
        os.path.join(output_dir, "behavior_label_summary.csv"),
        index=False
    )

    print("\nSaved outputs")
    print("-" * 60)
    print(generations_path)
    print(os.path.join(output_dir, "summary_by_temperature.csv"))
    print(os.path.join(output_dir, "summary_by_prompt.csv"))
    print(os.path.join(output_dir, "behavior_label_summary.csv"))

    print("\nSummary by temperature")
    print("-" * 60)
    print(summary_by_temperature)

    print("\nBehavior label summary")
    print("-" * 60)
    print(behavior_label_summary)

In [ ]:
config = ExperimentConfig()

df = run_experiment(config)
create_summaries(df, config.output_dir)